Cell 1 — install + auth:

In [ ]:
!pip install -U google-genai -q

from google.colab import userdata
from google import genai

client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))
print("Gemini client ready")

Gemini client ready


In [ ]:
from pydantic import BaseModel

class RoomAnalysis(BaseModel):
    room_type: str
    furniture: list[str]
    decor: list[str]
    colors: list[str]
    style: str
    lighting: str
    spatial_observations: list[str]

room = RoomAnalysis(
    room_type="Bedroom",
    furniture=["bed", "table", "nightstand"],
    decor=["paintings", "wooden wardrobe"],
    colors=["beige", "brown"],
    style="Traditional",
    lighting="table lamps",
    spatial_observations=["bed against the wall", "nightstand on the left side"]
)

print(room)

room_type='Bedroom' furniture=['bed', 'table', 'nightstand'] decor=['paintings', 'wooden wardrobe'] colors=['beige', 'brown'] style='Traditional' lighting='table lamps' spatial_observations=['bed against the wall', 'nightstand on the left side']


Cell 2 — DesignPlan schema:

In [ ]:
from pydantic import BaseModel

class FurnitureItem(BaseModel):
    item: str
    description: str
    action: str  # e.g. "replace", "add", "keep"

class DesignPlan(BaseModel):
    design_style: str
    color_palette: list[str]
    furniture_changes: list[FurnitureItem]
    layout_changes: list[str]
    generation_prompt: str

def parse_design_plan(raw_text: str) -> DesignPlan:
    cleaned = re.sub(r"^```(json)?\s*|\s*```$", "", raw_text.strip())
    data = json.loads(cleaned)
    return DesignPlan(**data)

Cell 3 — the planning function:

In [ ]:
def plan_design(room: RoomAnalysis, style: str, budget: str, preferred_colors: list[str]) -> DesignPlan:
    prompt = f"""You are an interior design planner. Given a room analysis and user preferences, create a design plan.

Room analysis:
{room.model_dump_json(indent=2)}

User preferences:
- Style: {style}
- Budget: {budget}
- Preferred colors: {', '.join(preferred_colors)}

Respond with ONLY a valid JSON object, no other text, in exactly this format:

{{
  "design_style": "string",
  "color_palette": ["color1", "color2"],
  "furniture_changes": [
    {{"item": "string", "description": "string", "action": "replace/add/keep"}}
  ],
  "layout_changes": ["string"],
  "generation_prompt": "string describing the redesigned room for an image generation model"
}}

Do not include any explanation, markdown formatting, or text outside the JSON object."""

    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt
    )

    return parse_design_plan(response.text)

In [ ]:
import time
import re
import json

def plan_design_with_retry(room, style, budget, preferred_colors, max_retries=5):
    for attempt in range(max_retries):
        try:
            return plan_design(room, style, budget, preferred_colors)
        except Exception as e:
            if attempt == max_retries - 1:
                raise
            wait = 5 * (attempt + 1)
            print(f"Attempt {attempt+1} failed ({e}), retrying in {wait}s...")
            time.sleep(wait)

design = plan_design_with_retry(
    room=room,
    style="modern minimalist",
    budget="$1500",
    preferred_colors=["beige", "white", "light wood"]
)
print(design)

design_style='Modern Minimalist' color_palette=['Beige', 'White', 'Light Wood'] furniture_changes=[FurnitureItem(item='Bed Frame', description='Low-profile light wood platform bed with clean lines.', action='replace'), FurnitureItem(item='Nightstand', description='Sleek light wood or white floating nightstand with hidden storage.', action='replace'), FurnitureItem(item='Right Nightstand', description='Matching light wood floating nightstand for visual balance on the right side of the bed.', action='add'), FurnitureItem(item='Table', description='Minimalist white writing desk with light wood legs.', action='replace'), FurnitureItem(item='Wooden Wardrobe', description='Modern white wardrobe with light wood trim and handleless doors.', action='replace')] layout_changes=['Center the bed frame on the primary wall to fit matching nightstands on both left and right sides.', 'Place the minimalist desk in a corner near natural light to keep the central room flow open.'] generation_prompt='A mod